# 01 - Análise Exploratória de Dados

Objetivo deste notebook: importar, ler e entender o conjunto de dados Instacart Market Basket Analysis antes de qualquer modelagem.

Nesta etapa vamos responder:

- Do que se trata a base de dados?
- Quais arquivos, colunas e relacionamentos existem?
- Qual é a qualidade inicial das variáveis?
- Existem nulos, duplicados ou problemas de integridade?
- Quais sinais parecem úteis para um sistema de recomendação?

## 1. Configuração inicial

Nesta etapa são carregadas as bibliotecas utilizadas no notebook e definidas opções de exibição do pandas. Também é criado o caminho base para localizar os arquivos brutos do Instacart, mantendo o notebook executável tanto a partir da pasta `notebooks` quanto da raiz do projeto.

In [3]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", "{:.2f}".format)

### Definição dos caminhos

O caminho dos dados brutos é centralizado em `RAW_DATA_DIR`. Essa definição evita caminhos fixos espalhados pelo notebook e facilita a reprodução da análise em outro ambiente.

In [5]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw" / "instacart"

RAW_DATA_DIR

PosixPath('/Users/cassiojr/FIAP/Tech Challenge 2/FIAP-fase2-e-commerce/data/raw/instacart')

## 2. Arquivos disponíveis

A base é composta por tabelas relacionais. Os arquivos maiores são os itens comprados em pedidos anteriores e a tabela de pedidos.

In [5]:
csv_files = sorted(RAW_DATA_DIR.glob("*.csv"))

file_inventory = pd.DataFrame(
    {
        "file_name": [file.name for file in csv_files],
        "size_mb": [file.stat().st_size / 1024**2 for file in csv_files],
    }
).sort_values("size_mb", ascending=False)

file_inventory

,file_name,size_mb
2,order_products__prior.csv,550.80
4,orders.csv,103.92
3,order_products__train.csv,23.54
5,products.csv,2.07
0,aisles.csv,0.00
1,departments.csv,0.00


## 3. Leitura dos dados

Os arquivos CSV são carregados em DataFrames separados, respeitando a estrutura original da base. Em seguida, eles são agrupados em um dicionário para facilitar comparações de volume, memória, tipos de dados e qualidade.

In [23]:
aisles = pd.read_csv(RAW_DATA_DIR / "aisles.csv")
departments = pd.read_csv(RAW_DATA_DIR / "departments.csv")
products = pd.read_csv(RAW_DATA_DIR / "products.csv")
orders = pd.read_csv(RAW_DATA_DIR / "orders.csv")
order_products_train = pd.read_csv(RAW_DATA_DIR / "order_products__train.csv")
order_products_prior = pd.read_csv(RAW_DATA_DIR / "order_products__prior.csv")

datasets = {
    "aisles": aisles,
    "departments": departments,
    "products": products,
    "orders": orders,
    "order_products_train": order_products_train,
    "order_products_prior": order_products_prior,
}

### Visão geral dos volumes

A tabela abaixo resume o tamanho de cada DataFrame em número de linhas, colunas e memória utilizada. Esse passo ajuda a identificar quais tabelas exigem mais atenção em joins, agregações e etapas futuras de modelagem.

In [24]:
overview = pd.DataFrame(
    {
        "dataset": name,
        "rows": len(df),
        "columns": df.shape[1],
        "memory_mb": df.memory_usage(deep=True).sum() / 1024**2,
    }
    for name, df in datasets.items()
).sort_values("rows", ascending=False)

overview

,dataset,rows,columns,memory_mb
5,order_products_prior,32434489,4,989.82
3,orders,3421083,7,332.71
4,order_products_train,1384617,4,42.26
2,products,49688,4,4.93
0,aisles,134,2,0.01
1,departments,21,2,0.00


## 4. Amostra e esquema dos dados

Nesta parte vamos observar colunas, tipos de dados e primeiras linhas para entender a função de cada tabela.

In [25]:
def describe_schema(dataframes: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """Resume colunas, tipos e quantidade de valores nulos por dataset."""
    rows = []

    for dataset_name, df in dataframes.items():
        for column in df.columns:
            rows.append(
                {
                    "dataset": dataset_name,
                    "column": column,
                    "dtype": str(df[column].dtype),
                    "nulls": df[column].isna().sum(),
                    "null_rate": df[column].isna().mean(),
                    "unique_values": df[column].nunique(dropna=True),
                }
            )

    return pd.DataFrame(rows)


schema = describe_schema(datasets)
schema

,dataset,column,dtype,nulls,null_rate,unique_values
0,aisles,aisle_id,int64,0,0.00,134
1,aisles,aisle,str,0,0.00,134
2,departments,department_id,int64,0,0.00,21
3,departments,department,str,0,0.00,21
4,products,product_id,int64,0,0.00,49688
5,products,product_name,str,0,0.00,49688
6,products,aisle_id,int64,0,0.00,134
7,products,department_id,int64,0,0.00,21
8,orders,order_id,int64,0,0.00,3421083
9,orders,user_id,int64,0,0.00,206209


### Inspeção visual das tabelas

A exibição das primeiras linhas complementa o resumo do esquema dos dados. Aqui é possível conferir o formato real dos registros, validar nomes de colunas e entender como as chaves se conectam entre as tabelas.

In [ ]:
for name, df in datasets.items():
    print(f"\n{name}")
    display(df.head())

## 5. O que cada tabela representa

- `orders`: pedidos por usuário, com ordem temporal, dia da semana, hora e dias desde o pedido anterior.
- `order_products__prior`: itens de pedidos anteriores. Esta tabela representa o histórico principal de comportamento.
- `order_products__train`: itens de pedidos no conjunto de treino. Pode ser usada como alvo de validação offline.
- `products`: catálogo de produtos, com ligação para `aisles` e `departments`.
- `aisles`: corredores/categorias intermediárias do catálogo.
- `departments`: departamentos principais do catálogo.

Para recomendação, o sinal central é a interação `user_id -> product_id`, obtida ao juntar `orders` com as tabelas `order_products`.

## 6. Qualidade dos dados

Vamos verificar nulos, duplicados e valores fora do domínio esperado.

### Valores ausentes

A primeira verificação de qualidade mede a quantidade e a proporção de valores nulos por coluna. Esse diagnóstico separa ausências esperadas, como o primeiro pedido de cada usuário, de possíveis problemas que precisariam de tratamento.

In [ ]:
missing_summary = (
    schema.loc[schema["nulls"] > 0]
    .sort_values(["null_rate", "nulls"], ascending=False)
    .reset_index(drop=True)
)

missing_summary

### Registros duplicados

A checagem de duplicidade confirma se há linhas repetidas nos arquivos carregados. Duplicatas poderiam distorcer contagens de compra, taxas de recompra e métricas de popularidade dos produtos.

In [ ]:
duplicate_summary = pd.DataFrame(
    {
        "dataset": name,
        "duplicate_rows": df.duplicated().sum(),
        "duplicate_rate": df.duplicated().mean(),
    }
    for name, df in datasets.items()
)

duplicate_summary

### Validação de domínio

Além de nulos e duplicados, algumas colunas possuem faixas de valores esperadas. Esta validação verifica dias da semana, horários, ordem dos pedidos, posição no carrinho e a variável binária de recompra.

In [ ]:
domain_checks = {
    "orders_invalid_order_dow": orders.loc[~orders["order_dow"].between(0, 6)].shape[0],
    "orders_invalid_order_hour": orders.loc[~orders["order_hour_of_day"].between(0, 23)].shape[0],
    "orders_invalid_order_number": orders.loc[orders["order_number"] < 1].shape[0],
    "prior_invalid_add_to_cart_order": order_products_prior.loc[order_products_prior["add_to_cart_order"] < 1].shape[0],
    "train_invalid_add_to_cart_order": order_products_train.loc[order_products_train["add_to_cart_order"] < 1].shape[0],
    "prior_invalid_reordered": order_products_prior.loc[~order_products_prior["reordered"].isin([0, 1])].shape[0],
    "train_invalid_reordered": order_products_train.loc[~order_products_train["reordered"].isin([0, 1])].shape[0],
}

pd.Series(domain_checks, name="invalid_rows").to_frame()

Observação esperada: `days_since_prior_order` deve possuir nulos no primeiro pedido de cada usuário, porque não existe pedido anterior.

In [ ]:
first_orders = orders["order_number"].eq(1)

pd.DataFrame(
    {
        "scenario": ["first_order", "not_first_order"],
        "rows": [first_orders.sum(), (~first_orders).sum()],
        "days_since_prior_order_nulls": [
            orders.loc[first_orders, "days_since_prior_order"].isna().sum(),
            orders.loc[~first_orders, "days_since_prior_order"].isna().sum(),
        ],
    }
)

## 7. Integridade relacional

Aqui validamos se as chaves de uma tabela encontram correspondência nas tabelas de referência.

### Validação das chaves

Como a base é relacional, é importante garantir que produtos, pedidos, corredores e departamentos estejam corretamente referenciados. Falhas nessa etapa poderiam gerar perdas ou inconsistências durante os joins.

In [26]:
integrity_checks = {
    "products_without_aisle": (~products["aisle_id"].isin(aisles["aisle_id"])).sum(),
    "products_without_department": (~products["department_id"].isin(departments["department_id"])).sum(),
    "prior_items_without_order": (~order_products_prior["order_id"].isin(orders["order_id"])).sum(),
    "train_items_without_order": (~order_products_train["order_id"].isin(orders["order_id"])).sum(),
    "prior_items_without_product": (~order_products_prior["product_id"].isin(products["product_id"])).sum(),
    "train_items_without_product": (~order_products_train["product_id"].isin(products["product_id"])).sum(),
}

pd.Series(integrity_checks, name="invalid_references").to_frame()

,invalid_references
products_without_aisle,0
products_without_department,0
prior_items_without_order,0
train_items_without_order,0
prior_items_without_product,0
train_items_without_product,0


## 8. Entendimento das variáveis principais

Com a qualidade e a integridade verificadas, a análise passa a observar as variáveis mais importantes para recomendação: sequência dos pedidos, horário da compra, intervalo entre compras e divisão entre conjuntos de avaliação.

### Estatísticas dos pedidos

As estatísticas descritivas resumem o comportamento temporal dos pedidos. Elas ajudam a observar frequência de compra, distribuição dos horários e possíveis limites naturais das variáveis.

In [27]:
orders[["order_number", "order_dow", "order_hour_of_day", "days_since_prior_order"]].describe()

,order_number,order_dow,order_hour_of_day,days_since_prior_order
count,3421083.00,3421083.00,3421083.00,3214874.00
mean,17.15,2.78,13.45,11.11
std,17.73,2.05,4.23,9.21
min,1.00,0.00,0.00,0.00
25%,5.00,1.00,10.00,4.00
50%,11.00,3.00,13.00,7.00
75%,23.00,5.00,16.00,15.00
max,100.00,6.00,23.00,30.00


In [28]:
orders["eval_set"].value_counts(normalize=True).rename("rate").to_frame().join(
    orders["eval_set"].value_counts().rename("rows")
)

,rate,rows
eval_set,,
prior,0.94,3214874
train,0.04,131209
test,0.02,75000


### Catálogo enriquecido

O catálogo de produtos é enriquecido com corredor e departamento. Essa visão torna as análises mais interpretáveis, pois substitui parte dos identificadores numéricos por categorias de negócio.

In [29]:
product_catalog = (
    products.merge(aisles, on="aisle_id", how="left")
    .merge(departments, on="department_id", how="left")
)

product_catalog.head()

,product_id,product_name,aisle_id,department_id,aisle,department
0,1,Chocolate Sandwich Cookies,61,19,cookies cakes,snacks
1,2,All-Seasons Salt,104,13,spices seasonings,pantry
2,3,Robust Golden Unsweetened Oolong Tea,94,7,tea,beverages
3,4,Smart Ones Classic Favorites Mini Rigatoni With Vodka Cream Sauce,38,1,frozen meals,frozen
4,5,Green Chile Anytime Sauce,5,13,marinades meat preparation,pantry


In [30]:
department_product_counts = (
    product_catalog["department"]
    .value_counts()
    .rename_axis("department")
    .reset_index(name="products")
)

department_product_counts.head(15)

,department,products
0,personal care,6563
1,snacks,6264
2,pantry,5371
3,beverages,4365
4,frozen,4007
5,dairy eggs,3449
6,household,3085
7,canned goods,2092
8,dry goods pasta,1858
9,produce,1684


## 9. Primeiros sinais para recomendação

Para o desafio, as compras anteriores serão tratadas como comportamento observado. O objetivo futuro será recomendar produtos com base no histórico do usuário, na frequência de recompra e em padrões agregados de usuários e produtos.

### Construção das interações históricas

As interações históricas unem os itens comprados anteriormente aos dados dos pedidos. A partir dessa tabela, cada linha passa a representar um produto comprado por um usuário em um pedido específico.

In [31]:
prior_interactions = order_products_prior.merge(
    orders[["order_id", "user_id", "order_number", "order_dow", "order_hour_of_day", "days_since_prior_order"]],
    on="order_id",
    how="left",
)

prior_interactions.head()

,order_id,product_id,add_to_cart_order,reordered,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2,33120,1,1,202279,3,5,9,8.00
1,2,28985,2,1,202279,3,5,9,8.00
2,2,9327,3,0,202279,3,5,9,8.00
3,2,45918,4,1,202279,3,5,9,8.00
4,2,30035,5,0,202279,3,5,9,8.00


### Produtos mais recorrentes

A contagem dos produtos mais comprados mostra quais itens aparecem com maior frequência no histórico. Esse é um primeiro sinal de popularidade e pode servir como baseline simples para recomendação.

In [33]:
top_products = (
    prior_interactions["product_id"]
    .value_counts()
    .head(20)
    .rename_axis("product_id")
    .reset_index(name="orders_with_product")
    .merge(product_catalog[["product_id", "product_name", "aisle", "department"]], on="product_id", how="left")
)

top_products

,product_id,orders_with_product,product_name,aisle,department
0,24852,472565,Banana,fresh fruits,produce
1,13176,379450,Bag of Organic Bananas,fresh fruits,produce
2,21137,264683,Organic Strawberries,fresh fruits,produce
3,21903,241921,Organic Baby Spinach,packaged vegetables fruits,produce
4,47209,213584,Organic Hass Avocado,fresh fruits,produce
5,47766,176815,Organic Avocado,fresh fruits,produce
6,47626,152657,Large Lemon,fresh fruits,produce
7,16797,142951,Strawberries,fresh fruits,produce
8,26209,140627,Limes,fresh fruits,produce
9,27845,137905,Organic Whole Milk,milk,dairy eggs


### Produtos com maior taxa de recompra

A taxa de recompra mede a proporção de vezes em que um produto foi comprado novamente. O filtro mínimo de ocorrências reduz ruído e evita destacar produtos com poucas compras e taxa artificialmente alta.

In [34]:
reorder_rate_by_product = (
    prior_interactions.groupby("product_id", as_index=False)
    .agg(
        orders_with_product=("order_id", "nunique"),
        reorder_rate=("reordered", "mean"),
    )
    .query("orders_with_product >= 100")
    .sort_values("reorder_rate", ascending=False)
    .head(20)
    .merge(product_catalog[["product_id", "product_name", "aisle", "department"]], on="product_id", how="left")
)

reorder_rate_by_product

,product_id,orders_with_product,reorder_rate,product_name,aisle,department
0,27740,101,0.92,Chocolate Love Bar,candy chocolate,snacks
1,35604,100,0.90,Maca Buttercups,candy chocolate,snacks
2,38251,111,0.89,Benchbreak Chardonnay,white wines,alcohol
3,10236,129,0.88,Fragrance Free Clay with Natural Odor Eliminator Cat Litter,cat food care,pets
4,20598,112,0.88,Thousand Island Salad Snax,fruit vegetable snacks,snacks
5,35496,451,0.86,Real2 Alkalized Water 500 ml,water seltzer sparkling water,beverages
6,9292,2921,0.86,Half And Half Ultra Pasteurized,milk,dairy eggs
7,45504,9108,0.86,Whole Organic Omega 3 Milk,milk,dairy eggs
8,43394,8477,0.86,Organic Lactose Free Whole Milk,soy lactosefree,dairy eggs
9,5514,3970,0.86,Organic Homogenized Whole Milk,milk,dairy eggs


## 10. Preparação da base para modelagem

Depois da exploração inicial, o notebook avança para a criação de uma base analítica. Nesta etapa, os datasets são carregados com nomes padronizados para separar a preparação de modelagem da análise exploratória anterior.

In [10]:
# Carregando os datasets necessários para criar o dataset principal.
dataset_orders = pd.read_csv(RAW_DATA_DIR / "orders.csv")
dataset_departments = pd.read_csv(RAW_DATA_DIR / "departments.csv")
dataset_aisles = pd.read_csv(RAW_DATA_DIR / "aisles.csv")
dataset_products = pd.read_csv(RAW_DATA_DIR / "products.csv")
dataset_order_products_prior = pd.read_csv(RAW_DATA_DIR / "order_products__prior.csv")
dataset_order_products_train = pd.read_csv(RAW_DATA_DIR / "order_products__train.csv")

### Conferência das amostras carregadas

Antes de criar variáveis derivadas, são exibidas pequenas amostras de cada tabela. Essa conferência rápida confirma que os arquivos foram lidos corretamente e que as colunas esperadas estão disponíveis.

In [12]:
from IPython.display import display

quantidade_de_linhas = 2
print("Orders")
display(dataset_orders.head(quantidade_de_linhas))
print("Departments")
display(dataset_departments.head(quantidade_de_linhas))
print("Aisles")
display(dataset_aisles.head(quantidade_de_linhas))
print("Products")
display(dataset_products.head(quantidade_de_linhas))
print("Order Products Prior")
display(dataset_order_products_prior.head(quantidade_de_linhas))
print("Order Products Train")
display(dataset_order_products_train.head(quantidade_de_linhas))

Orders


,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,NaN
1,2398795,1,prior,2,3,7,15.00


Departments


,department_id,department
0,1,frozen
1,2,other


Aisles


...

,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13


Order Products Prior


,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1


Order Products Train


,order_id,product_id,add_to_cart_order,reordered
0,1,49302,1,1
1,1,11109,2,1


### Engenharia de atributos

As variáveis são agregadas em três níveis complementares: par usuário-produto, usuário e produto. Essa estrutura captura histórico individual, comportamento geral do cliente e popularidade ou recorrência de cada item.

### Unificação do histórico e do conjunto de treino

Para criar as variáveis de modelagem, o histórico de compras anteriores é unido aos dados dos pedidos, trazendo o `user_id` e a ordem temporal de cada compra. O conjunto `train` também é convertido para pares `user_id` e `product_id`, que serão usados para marcar o alvo da recomendação.

In [ ]:
dataset_all_orders = dataset_order_products_prior.merge(
    dataset_orders[
        [
            'order_id',
            'user_id',
            'order_number',
            'order_dow',
            'order_hour_of_day',
            'days_since_prior_order',
        ]
    ],
    on='order_id',
    how='left',
)

train_user_products = (
    dataset_order_products_train
    .merge(dataset_orders[['order_id', 'user_id']], on='order_id', how='left')
    [['user_id', 'product_id']]
    .drop_duplicates()
)

In [34]:
# Agregação por usuário-produto (características do histórico)
user_product_features = (
    dataset_all_orders
    .groupby(['user_id', 'product_id'], as_index=False)
    .agg({
        'order_id': 'nunique',           # vezes que comprou
        'reordered': 'mean',              # taxa de recompra
        'add_to_cart_order': 'mean',      # posição média no carrinho
        'order_number': 'max',            # em qual pedido comprou
    })
    .rename(columns={
        'order_id': 'purchase_count',
        'reordered': 'reorder_rate',
        'add_to_cart_order': 'avg_cart_position',
        'order_number': 'last_order_number'
    })
)

# Agregação por usuário (comportamento geral)
user_features = (
    dataset_all_orders
    .groupby('user_id', as_index=False)
    .agg({
        'order_id': 'nunique',
        'product_id': 'nunique',
        'reordered': 'mean',
        'order_number': 'max',
    })
    .rename(columns={
        'order_id': 'user_total_orders',
        'product_id': 'user_unique_products',
        'reordered': 'user_reorder_rate',
        'order_number': 'user_total_purchases'
    })
)

# Agregação por produto (popularidade)
product_features = (
    dataset_all_orders
    .groupby('product_id', as_index=False)
    .agg({
        'user_id': 'nunique',
        'order_id': 'nunique',
        'reordered': 'mean',
    })
    .rename(columns={
        'user_id': 'product_unique_users',
        'order_id': 'product_total_orders',
        'reordered': 'product_reorder_rate'
    })
)

### Construção do dataset de treino

A base final combina as variáveis agregadas com informações do catálogo. O alvo (`target`) indica se o produto aparece no conjunto `train` para o mesmo usuário, permitindo treinar ou validar um modelo de recomendação.

In [35]:
# Tabela de treino: todos os pares usuário-produto do histórico com variáveis
training_data = (
    user_product_features
    .merge(user_features, on='user_id', how='left')
    .merge(product_features, on='product_id', how='left')
    .merge(dataset_products[['product_id', 'aisle_id', 'department_id']], on='product_id', how='left')
    .merge(dataset_aisles, on='aisle_id', how='left')
    .merge(dataset_departments, on='department_id', how='left')
)

# Criar target: produtos que aparecem em train = 1, outros = 0
train_set = (
    train_user_products
    .assign(target=1)
)

# Merge com train para saber quais foram recomprados
training_data = training_data.merge(
    train_set,
    on=['user_id', 'product_id'],
    how='left'
)

training_data['target'] = training_data['target'].fillna(0).astype('int8')

display(training_data.head())

,user_id,product_id,purchase_count,reorder_rate,avg_cart_position,last_order_number,user_total_orders,user_unique_products,user_reorder_rate,user_total_purchases,product_unique_users,product_total_orders,product_reorder_rate,aisle_id,department_id,aisle,department,target
0,1,196.00,10,0.90,1.40,10,11,18,0.69,11,8000,35791,0.78,77,7,soft drinks,beverages,1
1,1,10258.00,9,0.89,3.33,10,11,18,0.69,11,557,1946,0.71,117,19,nuts seeds dried fruit,snacks,1
2,1,10326.00,1,0.00,5.00,5,11,18,0.69,11,1923,5526,0.65,24,4,fresh fruits,produce,0
3,1,12427.00,10,0.90,3.30,10,11,18,0.69,11,1679,6476,0.74,23,19,popcorn jerky,snacks,0
4,1,13032.00,3,0.67,6.33,10,11,18,0.69,11,1286,3751,0.66,121,14,cereal,breakfast,1


## 11. Conclusão da análise exploratória

A análise confirmou a estrutura relacional da base Instacart, validou a qualidade inicial dos dados e destacou sinais úteis para recomendação, como frequência de compra, taxa de recompra, posição média no carrinho e popularidade dos produtos.

A partir desse ponto, o próximo passo natural é usar o dataset de treino para testar abordagens de recomendação e comparar métricas de desempenho offline.